# FASHN VTON 1.5 on free Colab T4

Put a **real garment photo** on a **person photo**. No Google AI Studio / Nano Banana quota.

If Makeo **Start Colab** sent you here: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**. Return to Makeo while the next cells install and download weights (several minutes). Come back to this tab when it asks for a person photo and a garment photo.

**Always open from GitHub** (this file updates on every push). Do **not** use *File → Save a copy in Drive* as the daily workflow — that copy goes stale.

- Latest `main`: [Open in Colab](https://colab.research.google.com/github/tmai-tech/Makeo/blob/explore-catalog-vton/notebooks/fashn_vton_colab.ipynb)
- Short link (after Pages deploy): [tmai-tech.github.io/Makeo/colab/fashn-vton/](https://tmai-tech.github.io/Makeo/colab/fashn-vton/)

| | |
|---|---|
| Weights | [fashn-ai/fashn-vton-1.5](https://huggingface.co/fashn-ai/fashn-vton-1.5) (~2 GB + 244 MB parser) |
| Code | [fashn-AI/fashn-vton-1.5](https://github.com/fashn-AI/fashn-vton-1.5) |
| Browser demo (no Colab) | [HF Space](https://huggingface.co/spaces/fashn-ai/fashn-vton-1.5) |
| License | **Apache 2.0** (commercial OK) |
| Output | 576×864 (2:3) |
| VRAM | ~8 GB claimed; T4 has ~15 GB |

**Before you run:** Runtime → Change runtime type → **T4 GPU** → Save.

First-look alternative (no install): [IDM-VTON Space](https://huggingface.co/spaces/yisol/IDM-VTON) — easier UI, **CC-BY-NC-SA-4.0 (not commercial)**.

## 1. Check the T4

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Runtime → Change runtime type → T4 GPU → Save, then Runtime → Restart session."
print("torch", torch.__version__, "cuda", torch.version.cuda)
print(torch.cuda.get_device_name(0), "cc", torch.cuda.get_device_capability(0))
print("T4 is Turing (7.5): pipeline will run fp32, not bf16. That is expected.")

## 2. Install (keep Colab's torch)

Do **not** `pip install -e .` with default deps — that can rebuild torch. Install the package without deps, then the extras. Use **CPU onnxruntime** for DWPose so we avoid Colab CUDA / ORT mismatches. The try-on model still runs on the T4.

In [ ]:
import os, sys, subprocess
from pathlib import Path

REPO = Path("/content/fashn-vton-1.5")
SRC = REPO / "src"

def sh(cmd):
    print("+", " ".join(cmd))
    subprocess.check_call(cmd)

if not (SRC / "fashn_vton").is_dir():
    os.chdir("/content")
    if REPO.exists():
        import shutil
        shutil.rmtree(REPO)
    sh(["git", "clone", "--depth", "1", "https://github.com/fashn-AI/fashn-vton-1.5.git", str(REPO)])

sh([sys.executable, "-m", "pip", "install", "-e", str(REPO), "--no-deps"])
sh([sys.executable, "-m", "pip", "install",
    "safetensors", "huggingface_hub", "pillow", "opencv-python-headless",
    "tqdm", "einops", "matplotlib", "fashn-human-parser>=0.1.1", "onnxruntime"])

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
import fashn_vton
print("fashn_vton ready from", fashn_vton.__file__)

## 3. Download weights (~2.2 GB, once per runtime)

In [ ]:
!python scripts/download_weights.py --weights-dir /content/fashn-vton-1.5/weights
!ls -lh /content/fashn-vton-1.5/weights /content/fashn-vton-1.5/weights/dwpose

## 4. Load the pipeline once

In [ ]:
import gc, os, sys, subprocess, shutil
from pathlib import Path
import torch

REPO, SRC = Path("/content/fashn-vton-1.5"), Path("/content/fashn-vton-1.5/src")

if not (SRC / "fashn_vton").is_dir():
    print("Package missing — installing now (this is the Install cell).")
    os.chdir("/content")
    if REPO.exists():
        shutil.rmtree(REPO)
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/fashn-AI/fashn-vton-1.5.git", str(REPO)])
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-e", str(REPO), "--no-deps"])
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "safetensors", "huggingface_hub", "pillow", "opencv-python-headless",
        "tqdm", "einops", "matplotlib", "fashn-human-parser>=0.1.1", "onnxruntime"])

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
from fashn_vton import TryOnPipeline

weights_dir = REPO / "weights"
if not (weights_dir / "model.safetensors").is_file():
    print("Weights missing — downloading (~2 GB).")
    subprocess.check_call([sys.executable, str(REPO / "scripts/download_weights.py"),
                           "--weights-dir", str(weights_dir)])

gc.collect()
torch.cuda.empty_cache()
pipe = TryOnPipeline(weights_dir=str(weights_dir), device="cuda")
print("pipeline ready", torch.cuda.memory_allocated() / 1e9, "GB allocated")

## 5. Upload person + garment

Tips that match the official Space:

- One person, full body or 3/4, face visible, plain-ish background
- Garment: flat-lay or hanger on contrasting cloth; show pallu / border / embroidery
- Portrait ~2:3 (the model outputs **576×864**)

Indian category map (v1.5 only has three buckets):

| Outfit | `category` | `garment_photo_type` |
|---|---|---|
| Kurti, blouse, jacket | `tops` | `flat-lay` if product shot |
| Palazzo, salwar, skirt | `bottoms` | `flat-lay` |
| Saree, anarkali, gown | `one-pieces` | `flat-lay` |
| Lehenga (3-piece) | no native type — try `one-pieces` or skirt=`bottoms` | expect weak drape |

In [ ]:
import io
from google.colab import files
from PIL import Image

def load_upload(label):
    print(label)
    print("Choose ONE image in the file picker, then wait until the upload bar finishes.")
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError("No file arrived. Re-run this cell and pick an image (do not cancel the picker).")
    name, data = next(iter(uploaded.items()))
    if data is None:
        raise RuntimeError(f"{name} uploaded as empty. Re-run and pick a JPG/PNG/WEBP.")
    img = Image.open(io.BytesIO(data)).convert("RGB")
    print(f"  loaded {name}  {img.size[0]}x{img.size[1]}")
    return img

person = load_upload("1/2  PERSON — full body or 3/4, face visible")
garment = load_upload("2/2  GARMENT — flat-lay or hanger of the real outfit")
print("person", person.size, "garment", garment.size)
display(person.resize((288, 432)), garment.resize((288, 432)))

## 6. Run try-on

On T4 use **20 steps** first (faster, less VRAM). Use 30–50 if the first pass looks usable.
`segmentation_free=True` is the default and better for volume (lehenga / anarkali).

In [ ]:
CATEGORY = "one-pieces"       # tops | bottoms | one-pieces
PHOTO_TYPE = "flat-lay"       # model | flat-lay
STEPS = 20                    # 20 fast / 30 balanced / 50 quality (Space default)
GUIDANCE = 1.5
SEED = 42

result = pipe(
    person_image=person,
    garment_image=garment,
    category=CATEGORY,
    garment_photo_type=PHOTO_TYPE,
    num_samples=1,
    num_timesteps=STEPS,
    guidance_scale=GUIDANCE,
    seed=SEED,
    segmentation_free=True,
)
out = result.images[0]
out_path = "/content/fashn_vton_result.png"
out.save(out_path)
print("saved", out_path, out.size)
display(out)
from google.colab import files as colab_files
colab_files.download(out_path)

## 7. Score the SKU (do this before wiring Makeo)

Same saree / kurti on every run:

1. Print / zari / border match the real cloth?
2. Pallu and pleats (saree) or lehenga layers?
3. Blouse invented or preserved?
4. Face / body same as the person photo?
5. Color shift?

If print holds and only drape is messy, v1.5 is a useful **open** baseline. If the motif changes, do not put this in the Makeo worker as the catalog engine.

### Known limits (from the model card)

- Output is **576×864**, not marketplace 2K
- Long→short or bulky→slim swaps can leave traces of the original clothes
- Body shape can drift
- T4 is not Ampere: first run is slower than the advertised ~5s on H100 (expect tens of seconds to a couple of minutes)
- If you OOM: Runtime → Restart session, reload pipeline, use `STEPS = 20`, one image at a time